# Task 3: Broadcast Join Experiment (Query 1)

- This experiment compares normal joins vs. broadcast joins using the silver taxi-trips and silver taxi-zone Delta tables for Query 1.
- It verifies that forcing a broadcast join reduces shuffle operations and improves performance.

## Connect to Spark

In [1]:
import sys
from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query1')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 12:12:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Register the original Delta tables

In [2]:
spark.catalog.clearCache()
register_broadcast_join_sources(spark, register_zones=True, register_air_quality=False)
print('Registered Silver taxi trips and taxi zone views.')

26/09/20 12:12:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Registered Silver taxi trips and taxi zone views.


## Isolate the join strategy

- Automatic broadcasting is disabled to prevent Spark from optimizing the 'normal' query.
- AQE is disabled to ensure the plan remains static during the experiment.

In [3]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

{'spark.sql.adaptive.enabled': 'false',
 'spark.sql.autoBroadcastJoinThreshold': '-1'}

## Measure normal and broadcast joins

In [4]:
NORMAL_QUERY = 'query_01_broadcast_normal'
BROADCAST_QUERY = 'query_01_broadcast_forced'
REPETITIONS = 9

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

26/09/20 12:12:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Normal and broadcast results match within float tolerance.


## Normal-join physical plan
Expect `SortMergeJoin` here.

In [5]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

== Physical Plan ==
* Sort (17)
+- Exchange (16)
   +- * HashAggregate (15)
      +- Exchange (14)
         +- * HashAggregate (13)
            +- * Project (12)
               +- * SortMergeJoin Inner (11)
                  :- * Sort (5)
                  :  +- Exchange (4)
                  :     +- * Filter (3)
                  :        +- * ColumnarToRow (2)
                  :           +- Scan parquet  (1)
                  +- * Sort (10)
                     +- Exchange (9)
                        +- * Filter (8)
                           +- * ColumnarToRow (7)
                              +- Scan parquet  (6)


(1) Scan parquet 
Output [2]: [pickup_zone_id#7, pickup_date#19]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
PushedFilters: [IsNotNull(pickup_zone_id)]
ReadSchema: struct<pickup_zone_id:int>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [pickup_zone_id#7, pickup_date#19]

(3) Filter [codegen id : 1]
Input [2]: [pickup_zo

## Broadcast-join physical plan
Expect `BroadcastHashJoin` and `BroadcastExchange` here.

In [6]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

== Physical Plan ==
* Sort (14)
+- Exchange (13)
   +- * HashAggregate (12)
      +- Exchange (11)
         +- * HashAggregate (10)
            +- * Project (9)
               +- * BroadcastHashJoin Inner BuildRight (8)
                  :- * Filter (3)
                  :  +- * ColumnarToRow (2)
                  :     +- Scan parquet  (1)
                  +- BroadcastExchange (7)
                     +- * Filter (6)
                        +- * ColumnarToRow (5)
                           +- Scan parquet  (4)


(1) Scan parquet 
Output [2]: [pickup_zone_id#7, pickup_date#19]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
PushedFilters: [IsNotNull(pickup_zone_id)]
ReadSchema: struct<pickup_zone_id:int>

(2) ColumnarToRow [codegen id : 2]
Input [2]: [pickup_zone_id#7, pickup_date#19]

(3) Filter [codegen id : 2]
Input [2]: [pickup_zone_id#7, pickup_date#19]
Condition : isnotnull(pickup_zone_id#7)

(4) Scan parquet 
Output [2]: [location_id#2

## Benchmark summary

In [7]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'], 2),
}
summary

{'normal_samples_seconds': [5.762,
  5.374,
  5.133,
  4.893,
  4.184,
  4.503,
  4.33,
  4.407,
  4.493],
 'normal_median_seconds': 4.503,
 'broadcast_samples_seconds': [3.156,
  2.685,
  2.198,
  1.964,
  1.892,
  2.062,
  2.407,
  2.264,
  2.457],
 'broadcast_median_seconds': 2.264,
 'broadcast_speedup_factor': 1.99}

## Cleanup

In [8]:
spark.catalog.clearCache()
spark.stop()